# MODULE 8-V3 — Source-Only A/B/C Architecture Selection and Freeze

## Purpose

This module is the **development-stage gate immediately before the publication-facing 118-fold LOSO run**.

It implements the following protocol:

1. Reuse the frozen Module 6 cache and metadata.
2. Use **source subjects only** for architecture selection.
3. Compare:
   - **A:** Compact CNN + Transformer
   - **B:** A + Center Loss
   - **C:** B + delayed weak subject-adversarial GRL
4. Select the variant using **validation balanced accuracy only** (macro-F1 and parameter count are tie-breakers).
5. **Never load or evaluate an outer target/test fold.**
6. Save an immutable **freeze manifest** containing the selected variant and full selection configuration.
7. Fail closed if a target/test manifest is accidentally referenced.

> **Important:** This notebook is a development gate. It does not produce a final 118-fold LOSO result. Run the final LOSO only after the freeze manifest is created and reviewed.

### Research protocol

```text
FROZEN MODULES 1–7
        |
        v
source-only development pool
        |
  grouped inner CV
        |
   A / B / C
        |
validation-only selection
        |
  FREEZE VARIANT
        |
        v
publication-facing 118-fold LOSO
        |
        v
cross-dataset zero-calibration
```


## V3 changes from V2

- **No outer target accuracy is used for model selection.**
- The pilot does **not** read the 118-fold outer test indices.
- Selection is based on **source validation balanced accuracy**.
- The source development pool is split by **subject group**, never by epoch.
- Each inner fold fits normalization on inner-training subjects only.
- Center loss and domain adversarial loss are training-only objectives; validation is classification-only.
- The chosen architecture is written to a freeze manifest and can be required by the later LOSO runner.
- The notebook includes hard assertions intended to prevent accidental protocol regression.


In [ ]:
# ============================================================
# CELL 1 — ENVIRONMENT + PROJECT ARTIFACT DISCOVERY
# ============================================================
import os, json, math, time, random, hashlib, warnings
from dataclasses import dataclass, asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")

SEED = 20260826
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

candidates = []
if os.environ.get("CROSS_DATASET_MI_PROJECT_ROOT"):
    candidates.append(Path(os.environ["CROSS_DATASET_MI_PROJECT_ROOT"]).expanduser())
candidates.extend([
    Path.home() / "Project2" / "cross_dataset_mi_project",
    Path.cwd() / "cross_dataset_mi_project",
    Path.home() / "cross_dataset_mi_project",
])

PROJECT_ROOT = next((p for p in candidates if p.exists()), candidates[0])

MANIFEST_ROOT = PROJECT_ROOT / "manifests"
CACHE_ROOT = PROJECT_ROOT / "cache"
RESULTS_ROOT = PROJECT_ROOT / "results"

MODULE8_ROOT = RESULTS_ROOT / "module_8_v3_source_only_selection"
RUN_ROOT = MODULE8_ROOT / "runs"
CHECKPOINT_ROOT = MODULE8_ROOT / "checkpoints"
HISTORY_ROOT = MODULE8_ROOT / "histories"
SELECTION_ROOT = MODULE8_ROOT / "selection"

for p in [MODULE8_ROOT, RUN_ROOT, CHECKPOINT_ROOT, HISTORY_ROOT, SELECTION_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

CACHE_PATH = CACHE_ROOT / "module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5"
CACHE_META_PATH = MANIFEST_ROOT / "module_6_cache_metadata.csv"
WITHIN_LOSO_PATH = MANIFEST_ROOT / "module_6_within_dataset_loso_folds.csv"
TRANSFER_PATH = MANIFEST_ROOT / "module_6_cross_dataset_transfer_folds.csv"
PROTOCOL_PATH = MANIFEST_ROOT / "module_6_baseline_protocol.json"

print("=" * 82)
print("MODULE 8-V3 — SOURCE-ONLY ARCHITECTURE SELECTION")
print("=" * 82)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Device      :", DEVICE)
print("PyTorch     :", torch.__version__)
print("Seed        :", SEED)

required = [CACHE_PATH, CACHE_META_PATH, WITHIN_LOSO_PATH, TRANSFER_PATH, PROTOCOL_PATH]
for p in required:
    print(("✓ " if p.exists() else "✗ ") + str(p))
assert all(p.exists() for p in required), "Missing required Module 6 artifacts."


In [ ]:
# ============================================================
# CELL 2 — FROZEN TASK SPEC + V3 SELECTION CONFIGURATION
# ============================================================
PRIMARY_CLASSES = ["left", "right", "feet"]
CLASS_TO_ID = {"left": 0, "right": 1, "feet": 2}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}

N_CHANNELS = 22
N_SAMPLES = 640
N_CLASSES = 3
TARGET_SFREQ = 160.0

@dataclass
class SelectionConfig:
    # protocol
    seed: int = SEED
    development_dataset: str = "BCI-IV-2a"
    n_inner_splits: int = 3
    n_repeats: int = 2
    max_subjects_for_smoke: int = 0
    smoke_test: bool = False

    # execution
    resume: bool = True
    run_selection: bool = True
    save_checkpoints: bool = True
    save_histories: bool = True

    # primary selection metric
    primary_metric: str = "balanced_accuracy"
    secondary_metric: str = "macro_f1"

    # training
    batch_size: int = 64
    max_epochs: int = 40
    warmup_epochs: int = 10
    lr: float = 3e-4
    min_lr: float = 1e-5
    weight_decay: float = 3e-4
    patience: int = 7
    grad_clip: float = 1.0
    label_smoothing: float = 0.05

    # losses
    center_loss_weight: float = 0.005
    domain_loss_weight: float = 0.02
    grl_max_lambda: float = 0.05
    center_lr: float = 0.05

    # architecture
    temporal_channels_each: int = 16
    fusion_channels: int = 48
    spatial_channels: int = 64
    token_dim: int = 128
    patch_stride: int = 16
    transformer_layers: int = 2
    transformer_heads: int = 4
    transformer_ffn: int = 256
    transformer_dropout: float = 0.20
    embedding_dim: int = 128
    classifier_dropout: float = 0.30

    # augmentation — training only
    train_noise_std: float = 0.005
    train_amp_jitter: float = 0.03
    train_channel_dropout: float = 0.03
    train_time_mask_prob: float = 0.10
    train_time_mask_max: int = 24

CFG = SelectionConfig()

assert CFG.primary_metric == "balanced_accuracy"
assert CFG.secondary_metric == "macro_f1"
assert CFG.development_dataset in {"BCI-IV-2a", "EEGMMIDB"}

print(json.dumps(asdict(CFG), indent=2))


In [ ]:
# ============================================================
# CELL 3 — LOAD ONLY FROZEN CACHE METADATA
# ============================================================
cache_meta_df = pd.read_csv(CACHE_META_PATH)

assert len(cache_meta_df) == 9316
assert cache_meta_df["subject"].nunique() == 118
assert set(cache_meta_df["harmonized_class"].unique()) == set(PRIMARY_CLASSES)

DEV_META = cache_meta_df[
    cache_meta_df["dataset"].astype(str) == CFG.development_dataset
].copy()

assert len(DEV_META) > 0
assert DEV_META["subject"].nunique() >= max(3, CFG.n_inner_splits)

print("All cache epochs        :", len(cache_meta_df))
print("All subjects            :", cache_meta_df["subject"].nunique())
print("Development dataset    :", CFG.development_dataset)
print("Development epochs     :", len(DEV_META))
print("Development subjects   :", DEV_META["subject"].nunique())
print("Development class counts:")
print(DEV_META["harmonized_class"].value_counts().sort_index())

# This notebook deliberately does NOT load outer LOSO test indices for execution.
# We read only the metadata needed to define a development cohort.
assert WITHIN_LOSO_PATH.exists(), "Outer LOSO manifest must exist for later modules, but it is not read here."
print("\nOUTER TARGET INDICES: NOT LOADED ✓")


In [ ]:
# ============================================================
# CELL 4 — CACHE ACCESS + SOURCE-ONLY NORMALIZATION
# ============================================================
class HDF5Store:
    def __init__(self, path):
        self.path = Path(path)
        self.h5 = None

    def __enter__(self):
        self.h5 = h5py.File(self.path, "r")
        return self

    def __exit__(self, exc_type, exc, tb):
        if self.h5 is not None:
            self.h5.close()
        self.h5 = None

    def get_X(self, indices):
        indices = np.asarray(indices, dtype=np.int64)
        if len(indices) == 0:
            return np.empty((0, N_CHANNELS, N_SAMPLES), dtype=np.float32)
        order = np.argsort(indices)
        sorted_idx = indices[order]
        X_sorted = self.h5["X"][sorted_idx]
        inverse = np.argsort(order)
        return np.asarray(X_sorted[inverse], dtype=np.float32)

class SourceOnlyRobustNormalizer:
    def __init__(self, eps=1e-6):
        self.eps = float(eps)
        self.median_ = None
        self.iqr_ = None
        self.fitted_subjects_ = tuple()

    def fit(self, X_source, source_subjects):
        X_source = np.asarray(X_source, dtype=np.float64)
        source_subjects = [str(s) for s in source_subjects]
        assert X_source.ndim == 3 and X_source.shape[1:] == (N_CHANNELS, N_SAMPLES)
        assert len(source_subjects) == len(X_source)

        values = X_source.transpose(1, 0, 2).reshape(N_CHANNELS, -1)
        self.median_ = np.median(values, axis=1)
        q25 = np.percentile(values, 25, axis=1)
        q75 = np.percentile(values, 75, axis=1)
        self.iqr_ = np.maximum(q75 - q25, self.eps)
        self.fitted_subjects_ = tuple(sorted(set(source_subjects)))
        return self

    def transform(self, X):
        assert self.median_ is not None
        X = np.asarray(X, dtype=np.float32)
        return ((X - self.median_[None, :, None]) /
                self.iqr_[None, :, None]).astype(np.float32)

    def assert_target_excluded(self, target_subject):
        if str(target_subject) in self.fitted_subjects_:
            raise AssertionError(f"Normalization leakage: {target_subject}")

def compute_class_ids(meta):
    return np.asarray([CLASS_TO_ID[x] for x in meta["harmonized_class"]], dtype=np.int64)

def safe_indices(json_string):
    return np.asarray(json.loads(json_string), dtype=np.int64)


In [ ]:
# ============================================================
# CELL 5 — GROUPED INNER-CV SPLIT (NO EPOCH-LEVEL LEAKAGE)
# ============================================================
def make_grouped_inner_folds(meta_df, n_splits=3, n_repeats=2, seed=0):
    subjects = np.asarray(sorted(meta_df["subject"].astype(str).unique()))
    assert len(subjects) >= n_splits

    folds = []
    for repeat in range(n_repeats):
        rng = np.random.default_rng(seed + repeat)
        shuffled = subjects.copy()
        rng.shuffle(shuffled)

        gkf = GroupKFold(n_splits=n_splits)
        dummy_X = np.zeros(len(meta_df), dtype=np.int8)
        groups = meta_df["subject"].astype(str).to_numpy()

        # GroupKFold itself is deterministic. Repeated runs use a different
        # subject ordering so the validation partitions change across repeats.
        order = np.array([np.where(shuffled == s)[0][0] for s in groups], dtype=np.int64)
        for split_id, (train_pos, val_pos) in enumerate(
            gkf.split(dummy_X, None, groups=order)
        ):
            train_idx = meta_df.iloc[train_pos]["cache_index"].astype(int).to_numpy()
            val_idx = meta_df.iloc[val_pos]["cache_index"].astype(int).to_numpy()

            train_subjects = set(meta_df.iloc[train_pos]["subject"].astype(str))
            val_subjects = set(meta_df.iloc[val_pos]["subject"].astype(str))
            assert train_subjects.isdisjoint(val_subjects)

            folds.append({
                "repeat": int(repeat),
                "split": int(split_id),
                "train_indices": train_idx,
                "val_indices": val_idx,
                "train_subjects": sorted(train_subjects),
                "val_subjects": sorted(val_subjects),
            })
    return folds

inner_folds = make_grouped_inner_folds(
    DEV_META,
    n_splits=CFG.n_inner_splits,
    n_repeats=CFG.n_repeats,
    seed=CFG.seed,
)

print("Inner folds:", len(inner_folds))
for f in inner_folds:
    print(
        f"repeat={f['repeat']} split={f['split']} "
        f"train_subjects={len(f['train_subjects'])} "
        f"val_subjects={len(f['val_subjects'])}"
    )

assert all(set(f["train_subjects"]).isdisjoint(f["val_subjects"]) for f in inner_folds)


In [ ]:
# ============================================================
# V3 CELL 6 — TRAINING-ONLY EEG AUGMENTATION + DATASETS
# ============================================================
class EEGAugment:
    def __init__(self, noise_std, amp_jitter, channel_dropout, time_mask_prob, time_mask_max):
        self.noise_std = float(noise_std)
        self.amp_jitter = float(amp_jitter)
        self.channel_dropout = float(channel_dropout)
        self.time_mask_prob = float(time_mask_prob)
        self.time_mask_max = int(time_mask_max)
    def __call__(self, x):
        x = x.clone()
        if self.noise_std > 0:
            x = x + torch.randn_like(x) * self.noise_std
        if self.amp_jitter > 0:
            scale = 1.0 + (2 * torch.rand((x.shape[0], 1)) - 1.0) * self.amp_jitter
            x = x * scale
        if self.channel_dropout > 0 and torch.rand(()) < self.channel_dropout:
            c = int(torch.randint(0, x.shape[0], (1,)).item())
            x[c] = 0.0
        if self.time_mask_prob > 0 and torch.rand(()) < self.time_mask_prob:
            width = int(torch.randint(8, self.time_mask_max + 1, (1,)).item())
            width = min(width, x.shape[1])
            start = int(torch.randint(0, max(1, x.shape[1] - width + 1), (1,)).item())
            x[:, start:start+width] = 0.0
        return x

class ArrayEEGDataset(Dataset):
    def __init__(self, X, y, subjects, augment=None):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)
        self.subjects = np.asarray(subjects, dtype=np.int64)
        self.augment = augment
        assert self.X.shape[1:] == (N_CHANNELS, N_SAMPLES)
        assert len(self.X) == len(self.y) == len(self.subjects)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx])
        if self.augment is not None:
            x = self.augment(x)
        return x, torch.tensor(self.y[idx], dtype=torch.long), torch.tensor(self.subjects[idx], dtype=torch.long)

class TensorEEGOnlyDataset(Dataset):
    def __init__(self, X):
        self.X = np.asarray(X, dtype=np.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return torch.from_numpy(self.X[i])


In [ ]:
# ============================================================
# V3 CELL 7 — GRL, CENTER LOSS, ATTENTION POOLING
# ============================================================
class GradientReversalFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = float(lambd)
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None

def grad_reverse(x, lambd):
    return GradientReversalFn.apply(x, lambd)

class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim) * 0.02)
    def forward(self, features, labels):
        centers_batch = self.centers.index_select(0, labels)
        return 0.5 * ((features - centers_batch) ** 2).sum(dim=1).mean()

class AttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        hidden = max(16, dim // 2)
        self.score = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))
    def forward(self, tokens):
        weights = torch.softmax(self.score(tokens).squeeze(-1), dim=1)
        pooled = torch.sum(tokens * weights.unsqueeze(-1), dim=1)
        return pooled, weights

def variant_settings(variant, base_cfg):
    variant = str(variant).upper()
    assert variant in {"A", "B", "C"}
    c = replace(base_cfg)
    if variant == "A":
        c.center_loss_weight = 0.0
        c.domain_loss_weight = 0.0
        c.grl_max_lambda = 0.0
    elif variant == "B":
        c.center_loss_weight = base_cfg.center_loss_weight
        c.domain_loss_weight = 0.0
        c.grl_max_lambda = 0.0
    else:
        c.center_loss_weight = base_cfg.center_loss_weight
        c.domain_loss_weight = base_cfg.domain_loss_weight
        c.grl_max_lambda = base_cfg.grl_max_lambda
    return c


In [ ]:
# ============================================================
# ============================================================
# CELL 8 — COMPACT DG-CONVORELENET V2 (FROZEN ARCHITECTURE FAMILY)
# ============================================================
# ============================================================
class TemporalBranch(nn.Module):
    def __init__(self, out_channels, kernel_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, out_channels, kernel_size=(1, kernel_size), padding=(0, kernel_size // 2), bias=False),
            nn.BatchNorm2d(out_channels),
            nn.Tanh(),
            nn.Dropout2d(0.08),
        )
    def forward(self, x): return self.net(x)

class DGConvoReleNetV2(nn.Module):
    def __init__(self, num_classes=3, num_domains=2, cfg=CFG, use_domain=True):
        super().__init__()
        self.use_domain = bool(use_domain and cfg.domain_loss_weight > 0 and cfg.grl_max_lambda > 0)
        kernels = [15, 31, 63]
        self.temporal = nn.ModuleList([TemporalBranch(cfg.temporal_channels_each, k) for k in kernels])
        self.spatial = nn.Sequential(
            nn.Conv2d(cfg.fusion_channels, cfg.spatial_channels, kernel_size=(N_CHANNELS, 1), bias=False),
            nn.BatchNorm2d(cfg.spatial_channels),
            nn.Tanh(),
            nn.Dropout2d(0.10),
        )
        self.patch = nn.Sequential(
            nn.Conv2d(cfg.spatial_channels, cfg.token_dim, kernel_size=(1, cfg.patch_stride), stride=(1, cfg.patch_stride), bias=False),
            nn.BatchNorm2d(cfg.token_dim),
            nn.Tanh(),
        )
        max_tokens = N_SAMPLES // cfg.patch_stride
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.token_dim,
            nhead=cfg.transformer_heads,
            dim_feedforward=cfg.transformer_ffn,
            dropout=cfg.transformer_dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=cfg.transformer_layers)
        self.positional = nn.Parameter(torch.zeros(1, max_tokens, cfg.token_dim))
        nn.init.normal_(self.positional, std=0.01)
        self.attn_pool = AttentionPooling(cfg.token_dim)
        self.embedding = nn.Sequential(
            nn.Linear(cfg.token_dim, cfg.embedding_dim),
            nn.Tanh(),
            nn.LayerNorm(cfg.embedding_dim),
            nn.Dropout(cfg.classifier_dropout),
        )
        self.classifier = nn.Linear(cfg.embedding_dim, num_classes)
        if self.use_domain:
            self.domain_classifier = nn.Sequential(
                nn.Linear(cfg.embedding_dim, 64),
                nn.Tanh(),
                nn.Dropout(0.20),
                nn.Linear(64, num_domains),
            )
        else:
            self.domain_classifier = None

    def forward(self, x, grl_lambda=0.0):
        x = x.unsqueeze(1)
        x = torch.cat([b(x) for b in self.temporal], dim=1)
        x = self.spatial(x)
        x = self.patch(x).squeeze(2).transpose(1, 2)
        L = x.shape[1]
        x = x + self.positional[:, :L]
        x = self.transformer(x)
        pooled, attn = self.attn_pool(x)
        emb = self.embedding(pooled)
        logits = self.classifier(emb)
        domain_logits = None
        if self.use_domain:
            domain_logits = self.domain_classifier(grad_reverse(emb, grl_lambda))
        return {"logits": logits, "embedding": emb, "attention": attn, "domain_logits": domain_logits}

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for v in ["A", "B", "C"]:
    cv = variant_settings(v, CFG)
    m = DGConvoReleNetV2(3, num_domains=8, cfg=cv, use_domain=(v == "C")).to(DEVICE)
    with torch.no_grad():
        o = m(torch.randn(2, 22, 640, device=DEVICE))
    print(v, "params=", f"{count_parameters(m):,}", "logits=", tuple(o["logits"].shape), "emb=", tuple(o["embedding"].shape))
    del m, o


In [ ]:
# ============================================================
# V3 CELL 9 — BALANCED SUBJECT+CLASS SAMPLING + SEED + METRICS
# ============================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_loaders(X_train, y_train, s_train, X_val, y_val, s_val, cfg):
    aug = EEGAugment(cfg.train_noise_std, cfg.train_amp_jitter, cfg.train_channel_dropout, cfg.train_time_mask_prob, cfg.train_time_mask_max)
    train_ds = ArrayEEGDataset(X_train, y_train, s_train, augment=aug)
    val_ds = ArrayEEGDataset(X_val, y_val, s_val, augment=None)

    # Balance both source subjects and classes.
    # This prevents high-trial subjects/classes from dominating training.
    pairs = pd.DataFrame({"s": np.asarray(s_train), "y": np.asarray(y_train)})
    pair_counts = pairs.groupby(["s", "y"]).size().to_dict()
    subject_counts = pairs.groupby("s").size().to_dict()
    weights = []
    for s, y in zip(s_train, y_train):
        w_subject = 1.0 / max(1, subject_counts[int(s)])
        w_class = 1.0 / max(1, pair_counts[(int(s), int(y))])
        weights.append(w_subject * w_class)
    weights = np.asarray(weights, dtype=np.float64)
    weights = weights / weights.mean()
    sampler = WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), num_samples=len(weights), replacement=True)
    pin = DEVICE.type == "cuda"
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=sampler, shuffle=False, num_workers=0, pin_memory=pin, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size * 2, shuffle=False, num_workers=0, pin_memory=pin)
    return train_loader, val_loader

def compute_all_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "confusion_matrix": json.dumps(cm.tolist()),
    }

def scheduled_grl_lambda(epoch, cfg):
    if cfg.grl_max_lambda <= 0 or epoch <= cfg.warmup_epochs:
        return 0.0
    p = (epoch - cfg.warmup_epochs) / max(1, cfg.max_epochs - cfg.warmup_epochs)
    p = min(max(p, 0.0), 1.0)
    # Smooth ramp starting from 0 after warm-up and approaching max_lambda.
    return float(cfg.grl_max_lambda * (2.0 / (1.0 + math.exp(-8.0 * (p - 0.5))) - 1.0))


In [ ]:
# ============================================================
# CELL 10 — SOURCE-ONLY ONE-FOLD TRAINER
# ============================================================
def train_one_source_fold(
    X_train, y_train, subj_train,
    X_val, y_val,
    cfg, variant, seed, fold_tag
):
    set_seed(seed)
    variant = str(variant).upper()
    assert variant in {"A", "B", "C"}

    use_domain = variant == "C"

    unique_train_subjects = sorted(set(map(int, subj_train)))
    subj_to_domain = {s: i for i, s in enumerate(unique_train_subjects)}
    domain_train = np.asarray(
        [subj_to_domain[int(s)] for s in subj_train],
        dtype=np.int64
    )
    domain_count = max(2, len(unique_train_subjects))

    train_loader, val_loader = make_loaders(
        X_train, y_train, subj_train,
        X_val, y_val, np.zeros(len(y_val), dtype=np.int64),
        cfg
    )

    model = DGConvoReleNetV2(
        N_CLASSES, domain_count, cfg=cfg, use_domain=use_domain
    ).to(DEVICE)
    center = CenterLoss(N_CLASSES, cfg.embedding_dim).to(DEVICE)

    model_optim = torch.optim.AdamW(
        model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay
    )
    center_optim = torch.optim.SGD(center.parameters(), lr=cfg.center_lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        model_optim, T_max=cfg.max_epochs, eta_min=cfg.min_lr
    )

    ce = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    dom_ce = nn.CrossEntropyLoss()

    best_state = None
    best_center = None
    best_score = -np.inf
    best_epoch = 0
    wait = 0
    history_rows = []

    for epoch in range(1, cfg.max_epochs + 1):
        model.train()
        center.train()

        grl = scheduled_grl_lambda(epoch, cfg) if use_domain else 0.0

        train_sum = 0.0
        train_n = 0
        train_correct = 0

        for xb, yb, db in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            if use_domain:
                db = torch.as_tensor(
                    [subj_to_domain[int(s)] for s in db.cpu().numpy()],
                    dtype=torch.long,
                    device=DEVICE
                )
            else:
                db = torch.zeros(len(xb), dtype=torch.long, device=DEVICE)

            model_optim.zero_grad(set_to_none=True)
            center_optim.zero_grad(set_to_none=True)

            out = model(xb, grl_lambda=grl)

            cls_loss = ce(out["logits"], yb)
            c_loss = (
                center(out["embedding"], yb)
                if variant in {"B", "C"}
                else torch.zeros((), device=DEVICE)
            )
            d_loss = (
                dom_ce(out["domain_logits"], db)
                if use_domain
                else torch.zeros((), device=DEVICE)
            )

            loss = (
                cls_loss
                + cfg.center_loss_weight * c_loss
                + cfg.domain_loss_weight * d_loss
            )

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            nn.utils.clip_grad_norm_(center.parameters(), 1.0)
            model_optim.step()

            if variant in {"B", "C"}:
                center_optim.step()

            train_sum += float(loss.detach().cpu()) * len(xb)
            train_n += len(xb)
            train_correct += int(
                (out["logits"].argmax(1) == yb).sum().detach().cpu()
            )

        scheduler.step()

        # ---- Source validation only ----
        model.eval()
        center.eval()
        y_true, y_pred = [], []
        val_sum = 0.0

        with torch.no_grad():
            for xb, yb, _ in val_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)

                out = model(xb, grl_lambda=0.0)
                cls_loss = ce(out["logits"], yb)
                c_loss = (
                    center(out["embedding"], yb)
                    if variant in {"B", "C"}
                    else torch.zeros((), device=DEVICE)
                )
                val_loss = cls_loss + cfg.center_loss_weight * c_loss
                val_sum += float(val_loss.detach().cpu()) * len(xb)

                y_true.extend(yb.cpu().numpy().tolist())
                y_pred.extend(out["logits"].argmax(1).cpu().numpy().tolist())

        val_metrics = compute_all_metrics(
            np.asarray(y_true), np.asarray(y_pred)
        )
        train_acc = train_correct / max(1, train_n)
        train_loss = train_sum / max(1, train_n)
        val_loss = val_sum / max(1, len(y_true))

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_metrics["accuracy"],
            "val_balanced_accuracy": val_metrics["balanced_accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "grl_lambda": grl,
        }
        history_rows.append(row)

        score = val_metrics["balanced_accuracy"]

        if score > best_score + 1e-8:
            best_score = score
            best_epoch = epoch
            wait = 0
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            best_center = {
                k: v.detach().cpu().clone()
                for k, v in center.state_dict().items()
            }
        else:
            wait += 1

        if epoch == 1 or epoch % 5 == 0 or wait == 0:
            print(
                f"[{fold_tag}] ep {epoch:03d} | "
                f"train={train_acc:.3f} | "
                f"val_bal={val_metrics['balanced_accuracy']:.3f} | "
                f"val_f1={val_metrics['macro_f1']:.3f} | "
                f"grl={grl:.4f}"
            )

        if wait >= cfg.patience:
            break

    assert best_state is not None
    model.load_state_dict(best_state)
    center.load_state_dict(best_center)

    history = pd.DataFrame(history_rows)
    best_row = history.loc[history["val_balanced_accuracy"].idxmax()]

    return (
        model,
        center,
        history,
        {
            "best_epoch": int(best_row["epoch"]),
            "best_val_balanced_accuracy": float(best_row["val_balanced_accuracy"]),
            "best_val_macro_f1": float(best_row["val_macro_f1"]),
            "best_val_accuracy": float(best_row["val_accuracy"]),
            "epochs_run": int(len(history)),
            "params": int(count_parameters(model)),
        },
    )


In [ ]:
# ============================================================
# CELL 11 — SOURCE-ONLY FOLD EXECUTOR
# ============================================================
def run_source_fold(fold, variant, repeat_idx, split_idx):
    variant = str(variant).upper()

    train_indices = np.asarray(fold["train_indices"], dtype=np.int64)
    val_indices = np.asarray(fold["val_indices"], dtype=np.int64)

    train_meta = cache_meta_df.loc[train_indices].copy()
    val_meta = cache_meta_df.loc[val_indices].copy()

    # Hard protocol guards.
    assert set(train_meta["dataset"]) == {CFG.development_dataset}
    assert set(val_meta["dataset"]) == {CFG.development_dataset}
    assert set(train_meta["subject"].astype(str)).isdisjoint(
        set(val_meta["subject"].astype(str))
    )

    with HDF5Store(CACHE_PATH) as store:
        X_train_raw = store.get_X(train_indices)
        X_val_raw = store.get_X(val_indices)

    normalizer = SourceOnlyRobustNormalizer().fit(
        X_train_raw,
        train_meta["subject"].astype(str).tolist()
    )

    # There is intentionally no target/test subject in this function.
    X_train = normalizer.transform(X_train_raw)
    X_val = normalizer.transform(X_val_raw)

    y_train = compute_class_ids(train_meta)
    y_val = compute_class_ids(val_meta)

    source_subjects = sorted(train_meta["subject"].astype(str).unique())
    subject_to_int = {s: i for i, s in enumerate(source_subjects)}
    s_train = np.asarray(
        [subject_to_int[str(s)] for s in train_meta["subject"]],
        dtype=np.int64
    )

    local_cfg = variant_settings(variant, CFG)
    fold_tag = (
        f"{variant} | DEV={CFG.development_dataset} | "
        f"R{repeat_idx+1}F{split_idx+1}"
    )

    start = time.time()

    model, center, history, best = train_one_source_fold(
        X_train, y_train, s_train,
        X_val, y_val,
        local_cfg, variant,
        CFG.seed + 10000 * repeat_idx + 100 * split_idx,
        fold_tag,
    )

    elapsed = time.time() - start

    row = {
        "variant": variant,
        "dataset": CFG.development_dataset,
        "repeat": int(repeat_idx),
        "split": int(split_idx),
        "train_subjects": len(source_subjects),
        "val_subjects": len(set(val_meta["subject"].astype(str))),
        "train_epochs": len(train_indices),
        "val_epochs": len(val_indices),
        "best_val_balanced_accuracy": best["best_val_balanced_accuracy"],
        "best_val_macro_f1": best["best_val_macro_f1"],
        "best_val_accuracy": best["best_val_accuracy"],
        "best_epoch": best["best_epoch"],
        "epochs_run": best["epochs_run"],
        "params": best["params"],
        "runtime_sec": elapsed,
        # Explicitly document what was NOT evaluated.
        "outer_target_evaluated": False,
        "outer_target_accuracy": np.nan,
        "outer_target_balanced_accuracy": np.nan,
        "outer_target_macro_f1": np.nan,
    }

    prefix = f"{variant}_dev_{CFG.development_dataset}_r{repeat_idx+1}_f{split_idx+1}"

    if CFG.save_histories:
        history.to_csv(HISTORY_ROOT / f"{prefix}.csv", index=False)

    if CFG.save_checkpoints:
        torch.save(
            {
                "model_state": model.state_dict(),
                "center_state": center.state_dict(),
                "config": asdict(local_cfg),
                "metadata": {
                    "variant": variant,
                    "protocol": "source_only_inner_cv",
                    "dataset": CFG.development_dataset,
                    "repeat": int(repeat_idx),
                    "split": int(split_idx),
                    "outer_target_evaluated": False,
                },
            },
            CHECKPOINT_ROOT / f"{prefix}.pt"
        )

    print(
        f"DONE | {fold_tag} | "
        f"val_bal={best['best_val_balanced_accuracy']:.4f} | "
        f"val_f1={best['best_val_macro_f1']:.4f} | "
        f"params={best['params']:,} | "
        f"time={elapsed/60:.1f} min"
    )
    return row


In [ ]:
# ============================================================
# CELL 12 — A/B/C SOURCE-ONLY SELECTION RUNNER
# ============================================================
SELECTION_RESULTS_PATH = SELECTION_ROOT / "source_only_ablation_results.csv"

def run_source_only_ablation():
    variants = ["A", "B", "C"]

    existing = pd.DataFrame()
    if SELECTION_RESULTS_PATH.exists() and CFG.resume:
        existing = pd.read_csv(SELECTION_RESULTS_PATH)

    done = set()
    if len(existing):
        for _, r in existing.iterrows():
            done.add(
                f"{r['variant']}::R{int(r['repeat'])}::F{int(r['split'])}"
            )

    rows = []

    for variant in variants:
        for fold in inner_folds:
            key = (
                f"{variant}::R{int(fold['repeat'])+1}"
                f"::F{int(fold['split'])+1}"
            )
            if key in done:
                print("SKIP existing:", key)
                continue

            row = run_source_fold(
                fold,
                variant=variant,
                repeat_idx=fold["repeat"],
                split_idx=fold["split"],
            )
            rows.append(row)

            existing = pd.concat(
                [existing, pd.DataFrame([row])],
                ignore_index=True
            )
            existing.to_csv(SELECTION_RESULTS_PATH, index=False)

    if SELECTION_RESULTS_PATH.exists():
        return pd.read_csv(SELECTION_RESULTS_PATH)

    return pd.DataFrame(rows)

selection_results = pd.DataFrame()

if CFG.run_selection:
    selection_results = run_source_only_ablation()
    print("Saved:", SELECTION_RESULTS_PATH)
else:
    print("Selection run disabled.")


In [ ]:
# ============================================================
# CELL 13 — SELECTION SUMMARY (VALIDATION ONLY)
# ============================================================
def summarize_selection(df):
    assert len(df) > 0

    out = (
        df.groupby("variant")
        .agg(
            val_balanced_mean=("best_val_balanced_accuracy", "mean"),
            val_balanced_std=("best_val_balanced_accuracy", "std"),
            val_f1_mean=("best_val_macro_f1", "mean"),
            val_f1_std=("best_val_macro_f1", "std"),
            val_accuracy_mean=("best_val_accuracy", "mean"),
            val_accuracy_std=("best_val_accuracy", "std"),
            params_mean=("params", "mean"),
            folds=("variant", "count"),
        )
        .reset_index()
    )

    return out

selection_summary = summarize_selection(selection_results)

display(
    selection_summary.sort_values(
        ["val_balanced_mean", "val_f1_mean", "params_mean"],
        ascending=[False, False, True]
    ).assign(
        val_balanced_mean_pct=lambda x: 100*x.val_balanced_mean,
        val_balanced_std_pct=lambda x: 100*x.val_balanced_std,
        val_f1_mean_pct=lambda x: 100*x.val_f1_mean,
        val_f1_std_pct=lambda x: 100*x.val_f1_std,
        val_accuracy_mean_pct=lambda x: 100*x.val_accuracy_mean,
    )[[
        "variant",
        "val_balanced_mean_pct",
        "val_balanced_std_pct",
        "val_f1_mean_pct",
        "val_f1_std_pct",
        "val_accuracy_mean_pct",
        "params_mean",
        "folds",
    ]]
)

assert len(selection_summary) == 3
assert set(selection_results["outer_target_evaluated"]) == {False}
assert selection_results[
    ["outer_target_accuracy", "outer_target_balanced_accuracy",
     "outer_target_macro_f1"]
].isna().all().all()

print("\nSTRICT SELECTION GUARD: PASS")
print("Outer target/test metrics were not used and remain NaN.")


In [ ]:
# ============================================================
# CELL 14 — PRE-REGISTERED VARIANT DECISION RULE
# ============================================================
def choose_variant(summary):
    s = summary.copy()

    # Primary: mean validation balanced accuracy.
    # Secondary: mean validation macro-F1.
    # Tertiary: lower mean parameter count.
    # Final deterministic tie-breaker: A < B < C.
    order = {"A": 0, "B": 1, "C": 2}
    s["order"] = s["variant"].map(order)

    s = s.sort_values(
        ["val_balanced_mean", "val_f1_mean", "params_mean", "order"],
        ascending=[False, False, True, True],
        kind="mergesort",
    )
    return str(s.iloc[0]["variant"]), s

SELECTED_VARIANT, ranked_summary = choose_variant(selection_summary)

print("=" * 82)
print("SOURCE-ONLY ARCHITECTURE DECISION")
print("=" * 82)
display(ranked_summary)

print(f"\nSELECTED_VARIANT = {SELECTED_VARIANT}")
print("Primary metric   = mean validation balanced accuracy")
print("Secondary metric = mean validation macro-F1")
print("Tie-breaker      = lower parameter count")


In [ ]:
# ============================================================
# CELL 15 — FREEZE MANIFEST + REPRODUCIBILITY HASH
# ============================================================
def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

selection_results_hash = sha256_file(SELECTION_RESULTS_PATH)
selection_summary_hash_input = json.dumps(
    ranked_summary.to_dict(orient="records"),
    sort_keys=True,
    default=str,
).encode("utf-8")
summary_hash = hashlib.sha256(selection_summary_hash_input).hexdigest()

freeze_manifest = {
    "module": "8-V3",
    "purpose": "source_only_architecture_selection_and_freeze",
    "selected_variant": SELECTED_VARIANT,
    "variants": {
        "A": "Compact CNN + Transformer",
        "B": "Compact CNN + Transformer + Center Loss",
        "C": "Compact CNN + Transformer + Center Loss + delayed weak subject GRL",
    },
    "development_dataset": CFG.development_dataset,
    "protocol": {
        "outer_target_evaluated": False,
        "outer_target_metrics_used_for_selection": False,
        "selection_metric_primary": CFG.primary_metric,
        "selection_metric_secondary": CFG.secondary_metric,
        "grouped_inner_cv": True,
        "n_inner_splits": CFG.n_inner_splits,
        "n_repeats": CFG.n_repeats,
        "source_only_normalization": True,
    },
    "training_config": asdict(variant_settings(SELECTED_VARIANT, CFG)),
    "selection_summary": ranked_summary.to_dict(orient="records"),
    "artifacts": {
        "selection_results": str(SELECTION_RESULTS_PATH),
        "selection_results_sha256": selection_results_hash,
        "summary_sha256": summary_hash,
    },
    "freeze_status": "LOCKED",
    "created_seed": CFG.seed,
    "next_module_requirement": (
        "The final 118-fold LOSO runner must load this manifest and use "
        "selected_variant exactly as recorded. It must not retune A/B/C."
    ),
}

FREEZE_PATH = SELECTION_ROOT / "module_8_v3_frozen_variant.json"
with open(FREEZE_PATH, "w", encoding="utf-8") as f:
    json.dump(freeze_manifest, f, indent=2)

print("Saved freeze manifest:", FREEZE_PATH)
print(json.dumps(freeze_manifest, indent=2))


In [ ]:
# ============================================================
# CELL 16 — FAIL-CLOSED PROTOCOL AUDIT
# ============================================================
# This cell deliberately checks that V3 cannot be mistaken for final LOSO.

assert freeze_manifest["freeze_status"] == "LOCKED"
assert freeze_manifest["protocol"]["outer_target_evaluated"] is False
assert freeze_manifest["protocol"]["outer_target_metrics_used_for_selection"] is False
assert freeze_manifest["protocol"]["grouped_inner_cv"] is True
assert freeze_manifest["protocol"]["source_only_normalization"] is True

# The V3 results file must not contain any populated target metrics.
target_metric_cols = [
    "outer_target_accuracy",
    "outer_target_balanced_accuracy",
    "outer_target_macro_f1",
]
if SELECTION_RESULTS_PATH.exists():
    chk = pd.read_csv(SELECTION_RESULTS_PATH)
    assert chk[target_metric_cols].isna().all().all()
    assert set(chk["outer_target_evaluated"].astype(bool)) == {False}

print("=" * 82)
print("MODULE 8-V3 QA")
print("=" * 82)
print("Frozen Module 6 cache          ✓")
print("Grouped source-only CV        ✓")
print("No outer target indices read  ✓")
print("No target metrics evaluated   ✓")
print("Selection metric pre-registered ✓")
print("Source-only normalization     ✓")
print("A/B/C comparison complete     ✓")
print("Variant freeze manifest       ✓")
print("FAIL-CLOSED protocol guards   ✓")
print("\nSTATUS: READY FOR FINAL LOSO")


## Recommended next execution

After reviewing the freeze manifest:

1. Do **not** change A/B/C, center-loss weight, GRL strength, architecture depth, dropout, augmentation, or optimizer settings.
2. Run the final 118-fold LOSO notebook using `module_8_v3_frozen_variant.json`.
3. The final LOSO notebook should select the **checkpoint epoch from source validation inside each outer fold**, but it must not retune the architecture.
4. Report the 118-fold target results only after the freeze is complete.
5. Run cross-dataset zero-calibration as a separate locked protocol.

### Expected final artifact chain

```text
Module 6 frozen cache
      ↓
Module 8-V3 freeze manifest
      ↓
Final 118-fold within-dataset LOSO
      ↓
Cross-dataset zero-calibration
      ↓
Per-subject/domain-shift diagnostics
```

### Interpretation rule

A/B/C validation performance in this notebook is **development evidence**, not the final headline result.

The final headline result must come from the later target-isolated 118-fold LOSO evaluation.
